# 피처 선택 재학습 보고서

## 실험 개요

- 데이터 구간: `2024-10-01` 이후 `master.parquet`
- 원본 피처 수: 104개
- 목표: 피처 수를 줄여 과적합 위험과 추론 부담을 낮추면서 검증 성능을 유지하거나 개선함
- 선택 기준: validation AP, validation 순수익, 피처 수 패널티를 합친 점수로 후보를 고른 뒤 test 구간에서 최종 확인함
- 모델: XGBoost histogram tree, depth 3, L1/L2 정규화, 시간순 train/validation/test split, horizon별 purge 적용

## 최종 선택 결과

| 구간 | 후보 | 피처 수 | test AP | test ROC-AUC | test net | test hit | coverage | 권장 |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| short_30m | top_32_importance | 32 | 0.5889 | 0.7849 | 0.5700% | 74.17% | 5.67% | 예 |
| short_4h | top_24_importance | 24 | 0.5184 | 0.7427 | 1.2149% | 71.02% | 4.31% | 예 |
| long_2d | top_48_importance | 48 | 0.3451 | 0.6572 | 1.9146% | 39.55% | 0.30% | 예 |
| long_30d | top_24_importance | 24 | 0.1707 | 0.6799 | -2.0298% | 15.10% | 58.41% | 아니오 |

## 제외 또는 비권장 구간

| 구간 | 제외 사유 |
| --- | --- |
| long_60d | long_60d: target class가 한쪽으로만 구성되어 학습할 수 없음 |

- 권장 여부는 test 순수익이 양수이고, 거래 수가 100건 이상이며, validation 양성 비율이 1% 이상인지 확인해 표시함.
- 비권장 구간은 모델 파일에 들어 있어도 실거래 기본 진입 신호로 쓰지 않는 것을 권장함.

## 선택된 피처 조합

| 구간 | 선택 피처 |
| --- | --- |
| short_30m | `ema_26_ratio`, `dist_high_60`, `volatility_60m`, `ema_12_ratio`, `liquidity_pool_pressure_up`, `realized_vol_120m`, `ret_5m`, `volatility_30m_rt`, `volatility_240m`, `ret_30m`, `rsi_14`, `dist_low_60`, `macd_signal_ratio`, `ret_3m`, `ret_1m`, `volatility_5m`, `realized_vol_240m`, `liquidity_pool_pressure_down`, `macd_ratio`, `body_pct`, `volatility_15m`, `ret_60m`, `ema_60_ratio`, `binance_ret_5m`, `binance_volume_rel_30`, `rsi_overbought`, `value_rel_30`, `upbit_binance_ret_spread_5m`, `binance_range_pct`, `value_z_120`, `volume_rel_120`, `value_rel_120` |
| short_4h | `liquidity_pool_pressure_up`, `dist_high_60`, `volatility_240m`, `realized_vol_240m`, `realized_vol_1440m`, `dist_low_60`, `volatility_60m`, `volatility_1440m`, `realized_vol_120m`, `ema_12_ratio`, `liquidity_pool_pressure_down`, `binance_volume_rel_30`, `binance_ret_1m`, `macd_hist_ratio`, `binance_ret_5m`, `upbit_binance_ret_spread_5m`, `upbit_binance_ret_spread_15m`, `upbit_binance_ret_spread_60m`, `ret_60m`, `value_rel_30`, `ema_60_ratio`, `ret_30m`, `macd_signal_ratio`, `binance_ret_60m` |
| long_2d | `volatility_240m`, `realized_vol_1440m`, `volatility_1440m`, `liquidity_pool_pressure_up`, `weekend_activity`, `round_figure_distance`, `dow_sin`, `dist_high_60`, `market_fx`, `dow_cos`, `market_fx_change_1440m`, `market_fx_change_60m`, `ema_1440_ratio`, `realized_vol_240m`, `hour_sin`, `kimp_real`, `ret_2880m`, `value_rel_1440`, `dist_low_60`, `kimp_z_1440`, `btc_volatility_30m`, `asia_session`, `value_z_1440`, `ret_1440m`, `binance_ret_30m`, `us_session`, `binance_range_pct`, `realized_vol_120m`, `binance_volume_rel_240`, `liquidity_pool_pressure_down`, `hour_cos`, `volatility_60m`, `eth_volatility_30m`, `ret_120m`, `ema_120_ratio`, `upbit_binance_ret_spread_240m`, `ret_60m`, `ema_240_ratio`, `btc_lead_lag_60m`, `ret_720m`, `volume_rel_120`, `range_pct`, `volatility_15m`, `value_z_120`, `volatility_30m_rt`, `macd_ratio`, `realized_vol_30m`, `ret_240m` |
| long_30d | `market_fx`, `kimp_real`, `round_figure_distance`, `kimp_velocity_240m`, `kimp_velocity_15m`, `market_fx_change_60m`, `market_fx_change_1440m`, `dow_cos`, `volatility_1440m`, `dow_sin`, `realized_vol_1440m`, `realized_vol_30m`, `eth_volatility_30m`, `weekend_activity`, `kimp_velocity_5m`, `ret_2880m`, `realized_vol_240m`, `btc_volatility_30m`, `realized_vol_120m`, `volatility_240m`, `ret_1440m`, `volatility_30m_rt`, `upbit_binance_ret_spread_240m`, `ret_720m` |

## 해석

- 전체 104개 피처를 항상 쓰는 대신 horizon별로 필요한 피처만 선택함.
- `top_k_importance` 후보는 전체 피처 예비 모델의 중요도 순위를 기준으로 만들었고, `low_latency_domain`, `behavior_domain`은 사람이 이해 가능한 도메인 묶음으로 비교함.
- 최종 모델은 선택된 후보의 피처만 사용해 train과 validation 구간을 합쳐 재학습했고, test 구간은 마지막 검증으로 남겨둠.
- `long_30d`, `long_60d`는 시장 국면 변화와 타깃 희소성 때문에 짧은 구간보다 불안정할 수 있으므로 실거래 기본 진입에는 보수적으로 해석하는 것이 좋음.

## 산출물

- metrics JSON: `reports/feature_selection_metrics.json`
- 로컬 모델 pkl: `models/feature_selected_realtime_model.pkl`
- GitHub에는 pkl이 아닌 재현 스크립트, metrics, 보고서만 올림.


In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path('reports/feature_selection_metrics.json').read_text(encoding='utf-8'))
metrics['metadata']['selected_model_path']
